### Required Libraries Import Karna

Is cell me hum project ke liye required libraries import kar rahe hain.

### Imported Libraries

- **StateGraph, START, END**
  - LangGraph workflow banane ke liye.
  - `START` workflow ka starting point hai.
  - `END` workflow ka ending point hai.

- **ChatOpenAI**
  - OpenAI model ko access karne ke liye.

- **load_dotenv()**
  - `.env` file se API keys load karta hai.

- **TypedDict**
  - Workflow ke state ka structure define karta hai.

- **Annotated**
  - LangGraph ko batata hai ki kisi field ko multiple nodes kaise update karenge.

- **BaseModel & Field**
  - Pydantic ki help se structured output schema banane ke liye.

- **operator**
  - State merge karte waqt special operations perform karne ke liye use hota hai.

Finally,

```python
load_dotenv()
```

`.env` file se API keys load kar deta hai.


In [6]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator
import os

In [7]:
load_dotenv()

True

### LLM Initialize Karna

```python
model = ChatOpenAI(model="gpt-4o-mini")
```

Is cell me hum Language Model initialize kar rahe hain.

Ab jab bhi hume AI se koi task karwana hoga, wahi model use hoga.

**Reasoning**

Ek hi model object create karna efficient hota hai.

Har function ke andar baar-baar model create karne ki zarurat nahi padti.

In [8]:
model = ChatGroq(model="llama-3.1-8b-instant", api_key=os.getenv("GROQ_API-KEY"))

### Structured Output Schema Banana

Is cell me hum Pydantic ki help se ek output schema define karte hain.

```python
class EvaluationSchema(BaseModel):
```

Model ko bataya ja raha hai ki output hamesha do fields me aana chahiye.

- feedback
- score

Example Output

```python
{
    "feedback":"Essay is well written...",
    "score":8
}
```

## Reasoning

Normally LLM text return karta hai.

Structured Output use karne se hume predictable JSON jaisa output milta hai jise code me directly use kar sakte hain.

Ye parsing errors ko bhi reduce karta hai.


In [9]:
class EvaluationSchema(BaseModel):

    feedback: str = Field(description='Detailed feedbackfor the essay')
    score: int = Field(description='Score out of 10', ge=0, le=10)

### Structured Model Banana

```python
structured_model = model.with_structured_output(EvaluationSchema)
```

Ab hum normal model ko ek structured model me convert kar dete hain.

Iske baad jab bhi prompt bhejenge, model automatically `EvaluationSchema` ke format me response dega.

Example

Instead of

```
The essay is good...
Score : 8
```

Directly milega

```python
output.feedback

output.score
```

Ye LangGraph workflows me kaafi useful hota hai.

In [10]:
structured_model = model.with_structured_output(EvaluationSchema)

In [11]:
essay = """India in the Age of AI
As the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.

India's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for All) with a focus on inclusive growth, aiming to leverage AI in healthcare, agriculture, education, and smart mobility.

One of the most promising applications of AI in India lies in agriculture, where predictive analytics can guide farmers on optimal sowing times, weather forecasts, and pest control. In healthcare, AI-powered diagnostics can help address India’s doctor-patient ratio crisis, particularly in rural areas. Educational platforms are increasingly using AI to personalize learning paths, while smart governance tools are helping improve public service delivery and fraud detection.

However, the path to AI-led growth is riddled with challenges. Chief among them is the digital divide. While metropolitan cities may embrace AI-driven solutions, rural India continues to struggle with basic internet access and digital literacy. The risk of job displacement due to automation also looms large, especially for low-skilled workers. Without effective skilling and re-skilling programs, AI could exacerbate existing socio-economic inequalities.

Another pressing concern is data privacy and ethics. As AI systems rely heavily on vast datasets, ensuring that personal data is used transparently and responsibly becomes vital. India is still shaping its data protection laws, and in the absence of a strong regulatory framework, AI systems may risk misuse or bias.

To harness AI responsibly, India must adopt a multi-stakeholder approach involving the government, academia, industry, and civil society. Policies should promote open datasets, encourage responsible innovation, and ensure ethical AI practices. There is also a need for international collaboration, particularly with countries leading in AI research, to gain strategic advantage and ensure interoperability in global systems.

India’s demographic dividend, when paired with responsible AI adoption, can unlock massive economic growth, improve governance, and uplift marginalized communities. But this vision will only materialize if AI is seen not merely as a tool for automation, but as an enabler of human-centered development.

In conclusion, India in the age of AI is a story in the making — one of opportunity, responsibility, and transformation. The decisions we make today will not just determine India’s AI trajectory, but also its future as an inclusive, equitable, and innovation-driven society."""

### Prompt Test Karna

Is cell me ek prompt banaya gaya hai.

Prompt LLM ko bol raha hai:

- Essay evaluate karo.
- Feedback do.
- Score out of 10 do.

Uske baad

```python
structured_model.invoke(prompt)
```

call kiya ja raha hai.

Ye sirf testing ke liye hai taaki check ho sake ki structured output sahi aa raha hai ya nahi.

In [12]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
structured_model.invoke(prompt).feedback

"The essay provides a comprehensive analysis of India's position in the age of AI, highlighting both the opportunities and challenges. It offers a nuanced discussion of the potential applications of AI in various sectors, including agriculture, healthcare, and education. However, the essay could benefit from more specific examples and data to support its claims. Additionally, the discussion on data privacy and ethics is somewhat superficial and could be expanded upon. Overall, the essay demonstrates a good understanding of the topic and presents a well-structured argument. However, it falls short of providing a truly exceptional analysis due to some of the limitations mentioned above."

### Workflow State Banana

```python
class UPSCState(TypedDict):
```

Ye poore workflow ki state hai.

State ke andar ye values hongi:

- essay
- language_feedback
- analysis_feedback
- clarity_feedback
- overall_feedback
- individual_scores
- avg_score

### Special Field

```python
individual_scores: Annotated[list[int], operator.add]
```

Ye bahut important line hai.

Teen alag nodes apna score return karenge.

Normally LangGraph confuse ho jayega ki list overwrite kare ya merge.

`operator.add`

LangGraph ko bolta hai ki

```
[8]

+

[9]

+

[7]

=

[8,9,7]
```

matlab overwrite nahi karna,

append karna hai.

Ye parallel workflows me bahut useful hota hai.

In [13]:
class UPSCState(TypedDict):

    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

#### Language Evaluation Node

Ye node sirf language quality evaluate karta hai.

Workflow:

1. Essay read karta hai.
2. Prompt banata hai.
3. LLM ko bhejta hai.
4. Feedback aur score receive karta hai.
5. State update karta hai.

Return

```python
{
    "language_feedback":...,
    "individual_scores":[score]
}
```

Notice karo ki score list ke form me return kiya gaya hai.

Reason:

Baad me multiple scores merge honge.

In [14]:
def evaluate_language(state: UPSCState):

    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

### Analysis Evaluation Node

Ye node essay ki analytical depth evaluate karta hai.

LLM check karega:

- Arguments kitne strong hain.
- Examples ache hain ya nahi.
- Critical thinking hai ya nahi.

Output:

- Analysis feedback
- Score

In [15]:
def evaluate_analysis(state: UPSCState):

    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

#### Clarity Evaluation Node

Ye node essay ki clarity evaluate karta hai.
Ye check karta hai:
- Thoughts clear hain ya nahi.
- Flow acha hai ya nahi.
- Reader easily samajh pa raha hai ya nahi.
Return:
- Clarity feedback
- Score

In [16]:
def evaluate_thought(state: UPSCState):

    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

### Final Evaluation Node

Ye workflow ka final step hai.

Isme:
- Teenon feedback ka summary banaya jata hai.
- Individual scores ka average nikala jata hai.

Final feedback aur average score return kiye jate hain.

In [17]:
def final_evaluation(state: UPSCState):

    # summary feedback
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content

    # avg calculate
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}
    

#### Build LangGraph Workflow

Is cell me pura workflow create kiya gaya hai.

Har evaluation function ko ek node banaya gaya hai.

Ye nodes baad me graph ke through execute honge.

#### Connect Workflow Nodes

Is cell me graph ke edges define kiye gaye hain.

Flow:

START

↓
Language Evaluation

↓
Analysis Evaluation

↓
Clarity Evaluation

↓
Final Evaluation

↓
END

Sabhi evaluation nodes parallel run karte hain aur final node me merge hote hain.

In [18]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

# edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [19]:
essay2 = """India and AI Time

Now world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.

India have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.

In farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.

But problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.

One more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.

India must all people together – govern, school, company and normal people. We teach AI and make sure AI not bad. Also talk to other country and learn from them.

If India use AI good way, we become strong, help poor and make better life. But if only rich use AI, and poor no get, then big bad thing happen.

So, in short, AI time in India have many hope and many danger. We must go right road. AI must help all people, not only some. Then India grow big and world say "good job India"."""

### Execute Workflow

Is cell me initial state create ki gayi hai jisme sirf essay diya gaya hai.

`workflow.invoke()` complete workflow execute karta hai.

Final output me milta hai:
- Language feedback
- Analysis feedback
- Clarity feedback
- Overall feedback
- Individual scores
- Average score

In [21]:
intial_state = {
    'essay': essay
}

workflow.invoke(intial_state)

{'essay': "India in the Age of AI\nAs the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.\n\nIndia's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for 